# Swimplaces playground

Scratch notebook pro rychle overeni parseru a importeru nad `source_data/sample_data.csv`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "src":
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

ROOT_DIR

In [ ]:
import django
from django.core.management import call_command

django.setup()
call_command("migrate", verbosity=0, interactive=False)

SAMPLE_CSV = ROOT_DIR / "source_data" / "sample_data.csv"
SAMPLE_CSV

## Nacteni 10 vzorku

In [ ]:
import csv

with SAMPLE_CSV.open(encoding="utf-8-sig", newline="") as sample_file:
    reader = csv.reader(sample_file, delimiter=";")
    header = next(reader)
    sample_rows = list(reader)

len(sample_rows), header

## Parser preview

Ukazka mapovani vzorku do import DTO.

In [ ]:
from places.services.parsers import parse_swimplace_row

parsed_rows = [parse_swimplace_row(row) for row in sample_rows]

([
    {
        "external_id": row.external_id,
        "import_id": row.import_id,
        "name": row.name,
        "category": row.category,
        "rating": row.rating,
        "dog_swimming": row.dog_swimming,
    }
    for row in parsed_rows
    if row is not None
])

## Import sample dat

Importer uklada do standardni lokalni DB. Druhe spusteni ma byt idempotentni: `created=0`, `updated=10`, pokud uz sample data existuji.

In [ ]:
from places.models import SwimPlace
from places.services.swimplace_importer import import_swim_places

first_summary = import_swim_places(SAMPLE_CSV)
second_summary = import_swim_places(SAMPLE_CSV)

{
    "first_run": first_summary,
    "second_run": second_summary,
    "total_db_count": SwimPlace.objects.count(),
}

## Kontrola ulozenych sample dat

In [ ]:
sample_external_ids = [row.external_id for row in parsed_rows if row is not None]

list(
    SwimPlace.objects.filter(external_id__in=sample_external_ids)
    .order_by("external_id")
    .values(
        "external_id",
        "import_id",
        "name",
        "category",
        "rating",
        "dog_swimming",
    )
)